In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import ComplementNB
import wandb


try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    wandb_api = user_secrets.get_secret("WANDB_API_KEY")

    wandb.login(key=wandb_api)
    USE_WANDB = True

except Exception:
    print("W&B login skipped.")
    USE_WANDB = False

In [ ]:
!pip install -q transformers datasets accelerate

## Loading dataset

In [ ]:
train=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

test=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## Exploratory Data Analysis (EDA)

Before building any machine learning models, we inspect the dataset to understand its structure, identify the available features, and verify that the training and test data are in the expected format.

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.info()

In [ ]:
train.shape

In [ ]:
train.iloc[0]

In [ ]:
question = train.loc[0, "prompt"]

print(question)

In [ ]:
print(train.loc[0, "A"])

In [ ]:
print(train.loc[0, "B"])

In [ ]:
print(train.loc[0, "C"])

## Text Preprocessing

The question prompt and all answer options are combined into a single text field. This provides complete context for the NLP models during training and inference.

In [ ]:
documents = [
    train.loc[0, "prompt"],
    train.loc[0, "A"],
    train.loc[0, "B"],
    train.loc[0, "C"],
    train.loc[0, "D"],
    train.loc[0, "E"]
]

## TF-IDF 

TF-IDF (Term Frequency–Inverse Document Frequency) converts text into numerical feature vectors by assigning higher weights to words that are important within a document while reducing the influence of commonly occurring words. These vectors are later used as input for the machine learning models.

In [ ]:
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(documents)

In [ ]:
print(tfidf.get_feature_names_out())

In [ ]:
print(tfidf_matrix.toarray()[:2])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

Before training supervised models, cosine similarity is used to measure how similar two text representations are in the TF-IDF feature space. This serves as a simple baseline for comparing document similarity.

In [ ]:
similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])
print(similarity)

In [ ]:
def combine_text(row):
    return (
        f"Question: {row['prompt']} "
        f"Option A: {row['A']} "
        f"Option B: {row['B']} "
        f"Option C: {row['C']} "
        f"Option D: {row['D']} "
        f"Option E: {row['E']}"
    )

In [ ]:
train["text"] = train.apply(combine_text, axis=1)
test["text"] = test.apply(combine_text, axis=1)

In [ ]:
print(train["text"].iloc[0])

In [ ]:
X = train["text"]
y = train["answer"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

In [ ]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

In [ ]:
print(X_train_tfidf.shape)
print(X_val_tfidf.shape)

In [ ]:

wandb.init(
    project="smart-mcq-solver",
    name="logistic-regression",
    config={
        "model": "Logistic Regression",
        "vectorizer": "TF-IDF",
        "max_features": 5000,
        "stop_words": "english",
        "random_state": 42,
        "max_iter": 1000
    }
)

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [ ]:
model.fit(X_train_tfidf, y_train)

In [ ]:
y_proba = model.predict_proba(X_val_tfidf)

print(y_proba.shape)
print(y_proba[:3])

In [ ]:
print(model.classes_)

In [ ]:
import numpy as np

top3_idx = np.argsort(y_proba, axis=1)[:, -3:][:, ::-1]

In [ ]:
top3_predictions = model.classes_[top3_idx]

print(top3_predictions[:10])

In [ ]:
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0

    for i, p in enumerate(predicted):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score


def mapk(actuals, predictions, k=3):
    return np.mean([
        apk(a, p, k)
        for a, p in zip(actuals, predictions)
    ])

In [ ]:
y_pred = model.predict(X_val_tfidf)

accuracy = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred, average="weighted")

print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:

score = mapk(y_val.tolist(), top3_predictions.tolist(), k=3)

print("MAP@3:", score)

In [ ]:
wandb.log({
    "Accuracy": accuracy,
    "F1 Score": f1,
    "MAP@3": score
})

wandb.finish()

## Naive Bayes

In [ ]:
nb_model = ComplementNB()

In [ ]:
wandb.init(
    project="smart-mcq-solver",
    name="Complement Naive Bayes",
    config={
        "model": "Complement Naive Bayes",
        "vectorizer": "TF-IDF"
    }
)

In [ ]:
nb_model.fit(X_train_tfidf, y_train)

In [ ]:
nb_pred = nb_model.predict(X_val_tfidf)

accuracy = accuracy_score(y_val, nb_pred)
f1 = f1_score(y_val, nb_pred, average="weighted")

print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
nb_proba = nb_model.predict_proba(X_val_tfidf)

top3_idx = np.argsort(nb_proba, axis=1)[:, -3:][:, ::-1]

top3_pred = nb_model.classes_[top3_idx]

In [ ]:
score = mapk(
    y_val.tolist(),
    top3_pred.tolist(),
    k=3
)

print("MAP@3:", score)

In [ ]:
wandb.log({
    "Accuracy": accuracy,
    "F1 Score": f1,
    "MAP@3": score
})

wandb.finish()

## BiLSTM

In [ ]:
import re
from collections import Counter


import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder

In [ ]:
label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)

print(label_encoder.classes_)

In [ ]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

In [ ]:
print(tokenize(X_train.iloc[0])[:20])

In [ ]:
counter = Counter()

for text in X_train:
    counter.update(tokenize(text))

In [ ]:
vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in counter.items():
    if count >= 2:
        vocab[word] = len(vocab)

In [ ]:
print("Vocabulary size:", len(vocab))

In [ ]:
MAX_LEN = 256

def encode_text(text):
    tokens = tokenize(text)

    ids = [
        vocab.get(word, vocab["<UNK>"])
        for word in tokens
    ]

    ids = ids[:MAX_LEN]

    if len(ids) < MAX_LEN:
        ids += [0] * (MAX_LEN - len(ids))

    return ids

In [ ]:
sample = encode_text(X_train.iloc[0])

print(sample[:30])
print(len(sample))

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
class MCQDataset(Dataset):

    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        x = torch.tensor(
            encode_text(self.texts.iloc[idx]),
            dtype=torch.long
        )

        if self.labels is not None:

            y = torch.tensor(
                self.labels[idx],
                dtype=torch.long
            )

            return x, y

        return x

In [ ]:
train_dataset = MCQDataset(X_train, y_train_enc)
val_dataset = MCQDataset(X_val, y_val_enc)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

In [ ]:
class BiLSTMClassifier(nn.Module):

    def __init__(self,
                 vocab_size,
                 embed_dim=128,
                 hidden_dim=128,
                 num_classes=5):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x):

        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        hidden = torch.cat(
            (hidden[-2], hidden[-1]),
            dim=1
        )

        hidden = self.dropout(hidden)

        return self.fc(hidden)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BiLSTMClassifier(
    vocab_size=len(vocab)
).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
print(model)

In [ ]:
def train_epoch(model, loader, criterion, optimizer):

    model.train()

    total_loss = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
wandb.init(
    project="smart-mcq-solver",
    name="BiLSTM (Scratch)",
    config={
        "model": "BiLSTM",
        "embedding_dim": 128,
        "hidden_dim": 128,
        "batch_size": 32,
        "learning_rate": 0.001
    }
)

In [ ]:
def evaluate(model, loader):

    model.eval()

    predictions = []
    probabilities = []
    labels = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)

            outputs = model(x)

            probs = torch.softmax(outputs, dim=1)

            pred = torch.argmax(probs, dim=1)

            predictions.extend(pred.cpu().numpy())
            probabilities.extend(probs.cpu().numpy())
            labels.extend(y.numpy())

    return predictions, probabilities, labels

In [ ]:
EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {loss:.4f}")

    wandb.log({
        "Epoch": epoch + 1,
        "Training Loss": loss
    })

In [ ]:
predictions, probabilities, labels = evaluate(
    model,
    val_loader
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(labels, predictions)

f1 = f1_score(
    labels,
    predictions,
    average="weighted"
)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
top3_idx = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]

top3_labels = label_encoder.inverse_transform(
    top3_idx.flatten()
).reshape(top3_idx.shape)

true_labels = label_encoder.inverse_transform(labels)

score = mapk(
    true_labels.tolist(),
    top3_labels.tolist(),
    k=3
)

print("MAP@3:", score)

In [ ]:
import torch
import json

torch.save(model.state_dict(), "bilstm_model.pt")

with open("vocab.json", "w") as f:
    json.dump(vocab, f)

import joblib
joblib.dump(label_encoder, "label_encoder.joblib")

print("BiLSTM artifacts saved!")

In [ ]:
wandb.log({
    "Accuracy": accuracy,
    "F1 Score": f1,
    "MAP@3": score
})

wandb.finish()

## Pretrained Model 

# DistilBERT

In [ ]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

In [ ]:
train_texts = X_train.tolist()
val_texts = X_val.tolist()

train_labels = y_train_enc.tolist()
val_labels = y_val_enc.tolist()

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "label": val_labels
})

In [ ]:
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=5
)

In [ ]:
wandb.init(
    project="smart-mcq-solver",
    name="DistilBERT",
    config={
        "model": "DistilBERT",
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5
    }
)

In [ ]:
training_args = TrainingArguments(
    output_dir="/tmp/distilbert_results",
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    report_to="wandb"
)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
predictions = trainer.predict(val_dataset)

logits = predictions.predictions

labels = predictions.label_ids

In [ ]:
predicted_classes = np.argmax(logits, axis=1)

accuracy = accuracy_score(
    labels,
    predicted_classes
)

f1 = f1_score(
    labels,
    predicted_classes,
    average="weighted"
)

print(f"Accuracy : {accuracy:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
top3_idx = np.argsort(logits, axis=1)[:, -3:][:, ::-1]

top3_labels = label_encoder.inverse_transform(
    top3_idx.flatten()
).reshape(top3_idx.shape)

true_labels = label_encoder.inverse_transform(labels)

score = mapk(
    true_labels.tolist(),
    top3_labels.tolist(),
    k=3
)

print("MAP@3:", score)

In [ ]:
wandb.log({
    "Accuracy": accuracy,
    "F1 Score": f1,
    "MAP@3": score
})

wandb.finish()

# Final Model Training and Submission

After comparing the performance of all models, the selected model is retrained using the complete training dataset. Predictions are then generated for the unseen test dataset and the top three answer choices are selected for Kaggle submission.

In [ ]:
X_full = train["text"]
y_full = train["answer"]

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_full_tfidf = vectorizer.fit_transform(X_full)

X_test_tfidf = vectorizer.transform(test["text"])

In [ ]:
final_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

final_model.fit(
    X_full_tfidf,
    y_full
)

In [ ]:
test_proba = final_model.predict_proba(
    X_test_tfidf
)

In [ ]:
top3_idx = np.argsort(
    test_proba,
    axis=1
)[:, -3:][:, ::-1]

classes = final_model.classes_

top3_predictions = classes[top3_idx]

In [ ]:
print(top3_predictions[:5])

In [ ]:
submission = pd.DataFrame({
    "id": test["id"],
    "prediction": [
        " ".join(pred)
        for pred in top3_predictions
    ]
})
submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully!")

In [ ]:
submission.head()

# Model Comparison

The performance of all implemented models is compared using Accuracy, Weighted F1 Score, and MAP@3. MAP@3 is the primary evaluation metric for this challenge and is used to select the final model for generating the Kaggle submission.

In [ ]:
import pandas as pd

results = {
    "Model": ["Logistic Regression", "Complement Naive Bayes", "BiLSTM (Scratch)", "DistilBERT (Pretrained)"],
    "Accuracy": [1.0000, 1.0000, 0.8025, 0.5500],
    "F1 Score": [1.0000, 1.0000, 0.8024, 0.5058],
    "MAP@3":    [1.0000, 1.0000, 0.8896, 0.6979]
}

comparison = pd.DataFrame(results).sort_values("MAP@3", ascending=False).reset_index(drop=True)
comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.plot(x="Model", y=["Accuracy", "F1 Score", "MAP@3"], kind="bar", figsize=(10,6))
plt.title("Model Comparison on Validation Set")
plt.xlabel("Model")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

Logistic Regression and Naive Bayes show near-perfect validation scores (Accuracy/MAP@3 ≈ 1.0), which is inconsistent with their simplicity relative to BiLSTM and DistilBERT. This strongly suggests overfitting on the small (2000-row), high-dimensional TF-IDF feature space rather than genuine generalization. This is supported by the actual Kaggle leaderboard score of 0.74, which sits far below the inflated validation numbers and much closer to what BiLSTM/DistilBERT's validation MAP@3 would predict.

## Error Analysis

In [ ]:
error_df = pd.DataFrame({
    "text": X_val.reset_index(drop=True),
    "true_answer": true_labels,
    "predicted_answer": label_encoder.inverse_transform(predicted_classes)
})

errors = error_df[error_df["true_answer"] != error_df["predicted_answer"]]

print(f"DistilBERT misclassified {len(errors)} out of {len(error_df)} validation examples")
errors.head(10)

In [ ]:
for i, row in errors.head(3).iterrows():
    print("Question + options:")
    print(row["text"][:400], "...")
    print(f"True answer: {row['true_answer']} | Predicted: {row['predicted_answer']}")
    print("-" * 80)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(true_labels, label_encoder.inverse_transform(predicted_classes), labels=label_encoder.classes_)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title("DistilBERT Confusion Matrix (Validation Set)")
plt.xlabel("Predicted Answer")
plt.ylabel("True Answer")
plt.tight_layout()
plt.show()